In [0]:
def test_inactive_customers(spark):

    from pyspark.sql.functions import col, max as _max, datediff, lit

    data = [
        ("C1", "2024-02-25"),
        ("C2", "2024-01-01"),
    ]

    columns = ["customer_id", "txn_date"]

    df = spark.createDataFrame(data, columns)

    last_txn_df = df.groupBy("customer_id") \
                    .agg(_max("txn_date").alias("last_txn_date"))

    # NOTE: This test assumes today's date > 2024-03-01
    result_df = last_txn_df.withColumn(
        "days_inactive",
        datediff(lit('2024-03-01'), col("last_txn_date"))
    ).filter(col("days_inactive") > 30)

    result = {r["customer_id"] for r in result_df.collect()}

    expected = {"C2"}

    try:
        assert result == expected
        print("✅ Test Passed")
    except AssertionError:
        print("❌ Test Failed")
        print("Expected:", expected)
        print("Got:", result)

test_inactive_customers(spark)